# Criação de Bancos de Dados (*Databases*) de Embeddings

Este notebook extrai os **embeddings** como resultados de um modelo e os salva em um **banco de dados SQLite** no seu Google Drive.

---

### O que este notebook faz (em ordem):
1. **Conecta ao seu Google Drive** para ler áudios e salvar resultados
2. **Instala as libs de software necessárias** automaticamente
3. **Carrega áudios** do Google Drive ou de um projeto do Arbimon
4. **Carrega um modelo backbone** do HuggingFace (Perch v2, BirdNET ou ONNX personalizado)
5. **Extrai o embedding** de cada segmento de áudio e salva em um banco de dados SQLite

### Modelos pré-configurados:
| Modelo | HuggingFace repo | Taxa de amostragem | Janela | Dimensão do embedding |
|---|---|---|---|---|
| Perch v2.0 | `biodiversica/Perch-onnx-backbone` | 32 000 Hz | 5,0 s | 1536 |
| BirdNET v2.4 | `biodiversica/BirdNET-onnx-backbone` | 48 000 Hz | 3,0 s | 1024 |

### Banco de dados de saída:
Os embeddings são salvos no arquivo SQLite que você especifica em **DB_PATH** (Parte 2).

Tabelas:
- `metadata` — parâmetros do modelo (`model_id`, `sample_rate`, `window_duration_s`, `overlap`, `embedding_size`, …)
- `embeddings` — uma linha por segmento: `(site_name, recording_id, chunk_index, datetime, data)`
- `sites` — coordenadas e IDs de stream por ponto de gravação (opcional)

O tempo de início é derivado do `chunk_index`, não armazenado diretamente:
`start_time = chunk_index × floor(window_duration_s × sample_rate × (1 − overlap)) / sample_rate`

### Como executar:
Execute cada célula **uma de cada vez**, de cima para baixo. Clique em ▶ ou pressione `Shift + Enter`.

> **Primeira vez utilizando notebooks interativos em python?** Uma célula com `[ ]` ainda não foi executada. Após executar aparece `[1]`, `[2]`, etc. Texto vermelho indica erro — leia a mensagem, ela geralmente indica exatamente o que corrigir.

---

Criado por [biodiversica](https://biodiversica.xyz). Para suporte ou feedback, abra uma *Issue* no [GitHub](https://github.com/biodiversica/bioacoustic-ipynbs/issues) ou entre em contato através do e-mail **info@biodiversica.xyz**

---
## Parte 1 — Conectar ao Google Drive e instalar libs

Execute as duas células abaixo. A primeira pedirá que você **permita acesso ao seu Google Drive** — clique no link e siga os passos.

In [ ]:
#@title 📂 Conectar ao Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive conectado com sucesso.')

In [ ]:
#@title 📦 Instalar libs { display-mode: "form" }

#@markdown **Dispositivo de computação** — onde o backbone ONNX será executado ao computar embeddings.
#@markdown - **CPU**: funciona em qualquer runtime do Colab (mais lento).
#@markdown - **GPU**: muito mais rápido, mas você precisa primeiro definir o runtime como GPU
#@markdown   (*Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*, depois execute novamente desde o início).
COMPUTE_DEVICE = 'CPU'  #@param ["CPU", "GPU"]

!pip install librosa soundfile huggingface_hub -q

# Instala a versão correspondente do ONNX Runtime. onnxruntime (CPU) e
# onnxruntime-gpu não podem coexistir, então removemos um antes de instalar o outro.
if COMPUTE_DEVICE == 'GPU':
    !pip uninstall -y onnxruntime onnxruntime-gpu -q
    # O Colab vem com CUDA 12, mas os wheels mais novos do onnxruntime-gpu são
    # compilados para CUDA 13 (procuram libcudart.so.13). Fixamos no último build CUDA 12.
    !pip install "onnxruntime-gpu==1.22.0" -q
else:
    !pip uninstall -y onnxruntime-gpu -q
    !pip install onnxruntime -q

print(f'\nTodas as libs instaladas com sucesso (dispositivo de computação: {COMPUTE_DEVICE}).')

---
## Parte 2 — Configuração

Preencha os formulários abaixo. **Não é necessário editar código** — basta digitar ou selecionar os valores em cada formulário e executar a célula.

Execute os três formulários de cima para baixo:
1. **Configurações Gerais** — pasta de saída, sobreposição
2. **Fonte de Áudio** — onde suas gravações estão armazenadas 
3. **Modelo** — qual *backbone* utilizar

> **Dica:** Os formulários ocultam o código automaticamente. Para ver o código por baixo, clique no ícone `{ }` no canto superior direito de qualquer célula de formulário.

In [ ]:
#@title ⚙️ Configurações Gerais { display-mode: "form" }

import os

#@markdown **Modo do banco de dados** — criar um novo banco ou atualizar um existente.
#@markdown Use *update existing* para adicionar embeddings a um banco já criado.
DB_MODE = 'create new'  #@param ["create new", "update existing"]

#@markdown **Caminho do banco de dados** — caminho completo para o arquivo `.db` no seu Google Drive.
#@markdown Para *create new*, este arquivo é criado (não será aceito um caminho para um arquivo existente).
#@markdown Para *update existing*, o arquivo deve existir e ser um banco de dados válido criado no formato deste notebook
DB_PATH = '/content/drive/MyDrive/Embeddings/meus_embeddings.embeddings.db'  #@param {type:"string"}

#@markdown ---
#@markdown **Sobreposição de segmentos (0.0–0.9)** — fração de sobreposição entre janelas de áudio consecutivas.
#@markdown 0.0 = sem sobreposição (padrão). 0.5 = 50% de sobreposição (cobertura mais densa, o dobro de segmentos por arquivo).
SEGMENT_OVERLAP = 0.0  #@param {type:"number"}
SEGMENT_OVERLAP = min(max(SEGMENT_OVERLAP, 0.0), 0.9)  # keep within 0.0–0.9

os.makedirs(os.path.dirname(DB_PATH) or '.', exist_ok=True)
print(f'Caminho do banco : {DB_PATH}')
print(f'Modo do banco    : {DB_MODE}')
print(f'Sobreposição     : {SEGMENT_OVERLAP}')

In [ ]:
#@title 🗂️ Fonte de Áudio { display-mode: "form" }

import os

#@markdown **Fonte** — onde estão armazenados seus arquivos de áudio.
AUDIO_SOURCE = 'google_drive'  #@param ["google_drive", "arbimon"]

#@markdown ---
#@markdown ### Configurações do Google Drive *(usado quando fonte = google_drive)*
#@markdown Caminho completo para a pasta com suas gravações de áudio no Google Drive.
DRIVE_INPUT_DIR = '/content/drive/MyDrive/audio'  #@param {type:"string"}

#@markdown **Prefixo do nome de arquivo** — deixe em branco para arquivos AudioMoth padrão (`YYYYMMDD_HHMMSS.wav`).
#@markdown Digite `SM4` para arquivos como `SM4_20250615_203000.wav`.
FILENAME_PREFIX = ''  #@param {type:"string"}

#@markdown **Filtro de data/hora** — processa apenas arquivos dentro desse intervalo.
#@markdown Deixe desativado para processar todos os arquivos.
FILTER_ENABLED = False  #@param {type:"boolean"}
FILTER_START_DATE = "2025-01-01"  #@param {type:"date"}
FILTER_START_TIME = "00:00"  #@param {type:"string"}
FILTER_END_DATE = "2025-12-31"  #@param {type:"date"}
FILTER_END_TIME = "23:59"  #@param {type:"string"}

#@markdown ---
#@markdown **Coordenadas por ponto** (opcional) — latitude e longitude por ponto de gravação.
#@markdown Formato: `NOME_PONTO=lat,lon` separados por ponto e vírgula.
#@markdown Exemplo: `PONTO_A=-15.1,-47.2;PONTO_B=-16.3,-48.1`
SITE_COORDINATES = ''  #@param {type:"string"}

FILENAME_PREFIX = FILENAME_PREFIX.strip()
_latlon_map = {}
for _e in SITE_COORDINATES.split(';'):
    _e = _e.strip()
    if '=' in _e:
        _n, _c = _e.split('=', 1)
        _p = _c.split(',')
        try:
            _latlon_map[_n.strip()] = (float(_p[0]), float(_p[1]))
        except Exception:
            pass

print(f'Fonte de áudio : {AUDIO_SOURCE}')
if AUDIO_SOURCE == 'google_drive':
    if not os.path.isdir(DRIVE_INPUT_DIR):
        print(f'AVISO: Pasta não encontrada: {DRIVE_INPUT_DIR}')
        print('Verifique o caminho acima — certifique-se de que o Google Drive está conectado.')
    else:
        print(f'Pasta de áudio : {DRIVE_INPUT_DIR}')
    if FILTER_ENABLED:
        print(f'Filtro de data : {FILTER_START_DATE} {FILTER_START_TIME} → {FILTER_END_DATE} {FILTER_END_TIME}')
    else:
        print('Filtro de data : desativado (todos os arquivos serão processados)')
elif AUDIO_SOURCE == 'arbimon':
    print('Fonte Arbimon selecionada — configure os pontos de gravação no Parte 3.')

In [ ]:
#@title 🤖 Modelo { display-mode: "form" }

import os

#@markdown **Modelo pré-configurado** — selecione um backbone de modelo de fundação.
#@markdown - **Perch v2**: backbone Google Perch (32 kHz · janela de 5 s · embeddings 1536-dim)
#@markdown - **BirdNET backbone**: backbone BirdNET v2.4 (48 kHz · janela de 3 s · embeddings 1024-dim)
#@markdown - **Custom ONNX**: use seu próprio backbone ONNX personalizado
MODEL_PRESET = 'Perch v2'  #@param ["Perch v2", "BirdNET backbone", "Custom ONNX"]

#@markdown ---
#@markdown ### Configurações do ONNX personalizado *(usado quando preset = Custom ONNX)*

#@markdown **Acesso ao modelo personalizado**
CUSTOM_MODEL_SOURCE = 'google_drive'  #@param ["google_drive", "huggingface"]

#@markdown **Caminho no Google Drive** — caminho completo para o arquivo `.onnx` backbone no Drive.
CUSTOM_DRIVE_MODEL_PATH = '/content/drive/MyDrive/models/my_backbone.onnx'  #@param {type:"string"}

#@markdown **HuggingFace repo ID** — ex.: `minha_org/meu-backbone-onnx`
CUSTOM_HF_REPO = ''  #@param {type:"string"}

#@markdown **Nome do arquivo no HuggingFace** — ex.: `backbone.onnx`
CUSTOM_HF_FILE = 'backbone.onnx'  #@param {type:"string"}

#@markdown **Taxa de amostragem (Hz)**
CUSTOM_SAMPLE_RATE = 48000  #@param {type:"integer"}

#@markdown **Duração da janela (segundos)** — comprimento de cada janela de áudio fornecida ao modelo.
CUSTOM_WINDOW_S = 3.0  #@param {type:"number"}

#@markdown **Nome do nó de entrada ONNX**
CUSTOM_INPUT_NAME = 'input'  #@param {type:"string"}

#@markdown **Nome do nó de saída ONNX**
CUSTOM_OUTPUT_NAME = 'embedding'  #@param {type:"string"}

#@markdown **Tamanho do embedding** — número de dimensões no vetor de saída.
CUSTOM_EMBEDDING_SIZE = 1024  #@param {type:"integer"}

_PRESETS = {
    'Perch v2': {
        'hf_repo':        'biodiversica/Perch-onnx-backbone',
        'hf_file':        'perch_v2_backbone.onnx',
        'sample_rate':    32000,
        'window_s':       5.0,
        'input_name':     'inputs',
        'output_name':    'embedding',
        'embedding_size': 1536,
        'model_name':     'perch_v2_backbone',
    },
    'BirdNET backbone': {
        'hf_repo':        'biodiversica/BirdNET-onnx-backbone',
        'hf_file':        'model_backbone.onnx',
        'sample_rate':    48000,
        'window_s':       3.0,
        'input_name':     'INPUT',
        'output_name':    'embedding',
        'embedding_size': 1024,
        'model_name':     'model_backbone',
    },
}

if MODEL_PRESET in _PRESETS:
    _p = _PRESETS[MODEL_PRESET]
    HF_REPO        = _p['hf_repo']
    HF_FILE        = _p['hf_file']
    SAMPLE_RATE    = _p['sample_rate']
    WINDOW_S       = _p['window_s']
    INPUT_NAME     = _p['input_name']
    OUTPUT_NAME    = _p['output_name']
    EMBEDDING_SIZE = _p['embedding_size']
    MODEL_NAME     = _p['model_name']
else:
    SAMPLE_RATE    = CUSTOM_SAMPLE_RATE
    WINDOW_S       = CUSTOM_WINDOW_S
    INPUT_NAME     = CUSTOM_INPUT_NAME
    OUTPUT_NAME    = CUSTOM_OUTPUT_NAME
    EMBEDDING_SIZE = CUSTOM_EMBEDDING_SIZE
    if CUSTOM_MODEL_SOURCE == 'huggingface':
        HF_REPO    = CUSTOM_HF_REPO.strip()
        HF_FILE    = CUSTOM_HF_FILE.strip()
        MODEL_NAME = os.path.splitext(HF_FILE)[0]
    else:
        HF_REPO    = None
        HF_FILE    = None
        MODEL_NAME = os.path.splitext(os.path.basename(CUSTOM_DRIVE_MODEL_PATH))[0]

WINDOW_SAMPLES = int(WINDOW_S * SAMPLE_RATE)

print(f'Modelo pré-configurado : {MODEL_PRESET}')
print(f'Nome do modelo         : {MODEL_NAME}')
print(f'Taxa de amostragem     : {SAMPLE_RATE} Hz')
print(f'Duração da janela      : {WINDOW_S} s  ({WINDOW_SAMPLES} amostras)')
print(f'Tamanho do embedding   : {EMBEDDING_SIZE}')
print(f'Sobreposição           : {SEGMENT_OVERLAP}')
if HF_REPO:
    print(f'HF repo                : {HF_REPO}')
    print(f'HF file                : {HF_FILE}')

---
## Parte 3 — Entrar no Arbimon *(ignore se for usar o Google Drive)*

Execute as células abaixo **apenas se** você definiu `AUDIO_SOURCE = "arbimon"` na Parte 2.
Se estiver usando o Google Drive, vá direto para a **Parte 4**.

**O que esperar no primeiro login:** uma URL e um código curto serão exibidos. Abra essa URL no navegador e insira o código para autorizar o acesso. Suas credenciais são salvas no Google Drive para execuções futuras.

In [ ]:
#@title 🔑 Entrar no Arbimon { display-mode: "form" }
CREDENTIALS_PATH = '/content/drive/MyDrive/rfcx/.rfcx_credentials'  #@param {type:"string"}

import os

# Instala o SDK do Arbimon aqui (não no topo) para que ele só seja baixado quando
# você realmente usar o Arbimon. Execute esta célula apenas se a origem do áudio for Arbimon.
!wget -q https://github.com/rfcx/rfcx-sdk-python/releases/download/0.3.1/rfcx-0.3.1-py3-none-any.whl -O /tmp/rfcx-0.3.1-py3-none-any.whl
!pip install /tmp/rfcx-0.3.1-py3-none-any.whl -q
import rfcx

os.makedirs(os.path.dirname(CREDENTIALS_PATH), exist_ok=True)

client = rfcx.Client()

if os.path.exists(CREDENTIALS_PATH):
    print('Arquivo de credenciais encontrado — fazendo login automaticamente...')
    client.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
else:
    print('Nenhuma credencial salva encontrada.')
    print('Uma URL será exibida abaixo. Abra-a no navegador e faça login com sua conta Arbimon.')
    print('-' * 70)
    client.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
    print('-' * 70)
    print(f'Credenciais salvas em: {CREDENTIALS_PATH}')
    print('Da próxima vez, o login será feito automaticamente.')

print('\nLogin no Arbimon realizado com sucesso.')

In [ ]:
#@title 🔍 Listar meus projetos e pontos no Arbimon { display-mode: "form" }
#@markdown Opcional — execute esta célula para encontrar os IDs de stream dos seus pontos de gravação.

print('Buscando seus projetos e pontos de gravação no Arbimon...')
print('=' * 70)

projects = client.projects()
if not projects:
    print('Nenhum projeto encontrado. Certifique-se de estar logado na conta correta.')
else:
    for project in projects:
        print(f"\nPROJETO: {project['name']}  (id: {project['id']})")
        print('  Pontos de gravação (streams):')
        try:
            streams = client.streams(projects=project['id'])
            if streams:
                for stream in streams:
                    print(f"    stream_id: '{stream['id']}',  nome: '{stream['name']}'")
            else:
                print('    (nenhum ponto encontrado neste projeto)')
        except Exception as e:
            print(f'    (não foi possível carregar os pontos: {e})')

In [ ]:
#@title 📍 Ponto de Gravação 1 { display-mode: "form" }

#@markdown **Stream ID** — o código curto do ponto de gravação (da célula opcional acima).
stream_id_1 = ''  #@param {type:"string"}
#@markdown **Data e hora de início** (horário local)
min_date_1 = "2025-12-01"  #@param {type:"date"}
min_time_1 = "17:00"  #@param {type:"string"}
#@markdown **Data e hora de fim** (horário local)
max_date_1 = "2025-12-01"  #@param {type:"date"}
max_time_1 = "18:00"  #@param {type:"string"}
if stream_id_1.strip():
    print(f"Ponto 1: '{stream_id_1}' — {min_date_1} {min_time_1} → {max_date_1} {max_time_1}")
else:
    print('Ponto 1: nenhum stream ID informado.')

### 📍 Pontos de gravação adicionais (opcional)
> Precisa analisar mais de um ponto? Expanda esta seção para configurar até 3 pontos adicionais.
> Deixe o **Stream ID** vazio em qualquer ponto que não precisar. Execute **Confirmar lista de pontos** novamente após adicionar pontos opcionais.

In [ ]:
#@title 📍 Ponto de Gravação 2 (opcional) { display-mode: "form" }
#@markdown Deixe o **Stream ID** vazio para pular este ponto.
stream_id_2 = ''  #@param {type:"string"}
min_date_2 = "2025-01-01"  #@param {type:"date"}
min_time_2 = "18:00"  #@param {type:"string"}
max_date_2 = "2025-01-02"  #@param {type:"date"}
max_time_2 = "06:00"  #@param {type:"string"}
if stream_id_2.strip():
    print(f"Ponto 2: '{stream_id_2}' — {min_date_2} {min_time_2} → {max_date_2} {max_time_2}")
else:
    print('Ponto 2: vazio — será ignorado.')

In [ ]:
#@title 📍 Ponto de Gravação 3 (opcional) { display-mode: "form" }
#@markdown Deixe o **Stream ID** vazio para pular este ponto.
stream_id_3 = ''  #@param {type:"string"}
min_date_3 = "2025-01-01"  #@param {type:"date"}
min_time_3 = "18:00"  #@param {type:"string"}
max_date_3 = "2025-01-02"  #@param {type:"date"}
max_time_3 = "06:00"  #@param {type:"string"}
if stream_id_3.strip():
    print(f"Ponto 3: '{stream_id_3}' — {min_date_3} {min_time_3} → {max_date_3} {max_time_3}")
else:
    print('Ponto 3: vazio — será ignorado.')

In [ ]:
#@title 📍 Ponto de Gravação 4 (opcional) { display-mode: "form" }
#@markdown Deixe o **Stream ID** vazio para pular este ponto.
stream_id_4 = ''  #@param {type:"string"}
min_date_4 = "2025-01-01"  #@param {type:"date"}
min_time_4 = "18:00"  #@param {type:"string"}
max_date_4 = "2025-01-02"  #@param {type:"date"}
max_time_4 = "06:00"  #@param {type:"string"}
if stream_id_4.strip():
    print(f"Ponto 4: '{stream_id_4}' — {min_date_4} {min_time_4} → {max_date_4} {max_time_4}")
else:
    print('Ponto 4: vazio — será ignorado.')

In [ ]:
#@title ✅ Confirmar lista de pontos de gravação { display-mode: "form" }
#@markdown Execute esta célula após preencher os formulários de pontos acima.

import datetime

_empty = ("", "2025-01-01", "00:00", "2025-01-01", "00:00")
try: stream_id_2
except NameError: stream_id_2, min_date_2, min_time_2, max_date_2, max_time_2 = _empty
try: stream_id_3
except NameError: stream_id_3, min_date_3, min_time_3, max_date_3, max_time_3 = _empty
try: stream_id_4
except NameError: stream_id_4, min_date_4, min_time_4, max_date_4, max_time_4 = _empty

def _build_job(stream_id, min_date_str, min_time_str, max_date_str, max_time_str):
    if not stream_id.strip():
        return None
    def _parse(date_str, time_str):
        d = datetime.date.fromisoformat(date_str.strip())
        parts = time_str.strip().split(':')
        h, m = int(parts[0]), int(parts[1])
        s = int(parts[2]) if len(parts) > 2 else 0
        return datetime.datetime(d.year, d.month, d.day, h, m, s)
    return {
        'stream_id':        stream_id.strip(),
        'stream_name':      stream_id.strip(),
        'min_date':         _parse(min_date_str, min_time_str),
        'max_date':         _parse(max_date_str, max_time_str),
        'lat':              '',
        'lon':              '',
        'timezone_str':     None,
        'utc_offset_hours': 0,
    }

def _tz_to_utc_offset(timezone_value, reference_dt=None):
    if timezone_value is None or timezone_value == "":
        return None
    try:
        return float(timezone_value)
    except (ValueError, TypeError):
        pass
    try:
        from zoneinfo import ZoneInfo
        import datetime as _dt
        tz = ZoneInfo(str(timezone_value))
        ref = (reference_dt or _dt.datetime.utcnow()).replace(tzinfo=_dt.timezone.utc)
        offset = ref.astimezone(tz).utcoffset()
        return offset.total_seconds() / 3600
    except Exception:
        return None

JOBS = []
for args in [
    (stream_id_1, min_date_1, min_time_1, max_date_1, max_time_1),
    (stream_id_2, min_date_2, min_time_2, max_date_2, max_time_2),
    (stream_id_3, min_date_3, min_time_3, max_date_3, max_time_3),
    (stream_id_4, min_date_4, min_time_4, max_date_4, max_time_4),
]:
    job = _build_job(*args)
    if job:
        JOBS.append(job)

_all_streams = client.streams() or []
_stream_map = {s['id']: s for s in _all_streams if isinstance(s, dict) and 'id' in s}
for job in JOBS:
    s = _stream_map.get(job['stream_id'], {})
    job['stream_name'] = s.get('name', job['stream_id']).replace(' ', '_')
    job['lat'] = s.get('latitude') or s.get('lat', '') or ''
    job['lon'] = s.get('longitude') or s.get('lon', '') or ''
    tz_val = s.get('timezone', None)
    job['timezone_str'] = str(tz_val) if tz_val else None
    mid_dt = job['min_date'] + (job['max_date'] - job['min_date']) / 2
    utc_off = _tz_to_utc_offset(tz_val, reference_dt=mid_dt)
    if utc_off is not None:
        job['utc_offset_hours'] = utc_off
    else:
        print(f"  AVISO: fuso horário não encontrado para {job['stream_name']} — usando UTC+0 como padrão")

if not JOBS:
    print('AVISO: Nenhum ponto de gravação configurado. Preencha pelo menos um formulário acima.')
else:
    print(f'{len(JOBS)} ponto(s) configurado(s):')
    for j in JOBS:
        utc_off = j['utc_offset_hours']
        print(f"  {j['stream_name']} ({j['stream_id']})")
        print(f"    {j['min_date'].strftime('%Y-%m-%d %H:%M')} → {j['max_date'].strftime('%Y-%m-%d %H:%M')}  (local UTC{utc_off:+.0f})")
    print('\nConfiguração concluída. Continue para o Parte 4.')

---
## Parte 4 — Procurar / baixar áudios

Esta célula acessa e verifica todos os arquivos de áudio a serem processados.

- **Google Drive**: procura a pasta configurada na Parte 2, respeitando qualquer filtro de data.
- **Arbimon**: baixa as gravações dos pontos configurados na Parte 3.

Os arquivos são organizados por ponto de gravação. O resultado é resumido abaixo — continue para a Parte 5 após revisar.

In [ ]:
#@title 🔍 Procurar / ⬇️ Baixar { display-mode: "form" }

import os
import re
import glob as _glob
from collections import defaultdict
from datetime import datetime

def _parse_file_datetime(filename, prefix=''):
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf'{re.escape(prefix)}_(\d{{8}})_(\d{{6}})', name)
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

def _get_site_name(audio_path, input_dir):
    parent = os.path.normpath(os.path.dirname(audio_path))
    root   = os.path.normpath(input_dir)
    if parent == root:
        return os.path.basename(root)
    return os.path.basename(parent)

all_audio_info = []  # list of {"path": str, "site": str, "dt": datetime|None}

if AUDIO_SOURCE == 'google_drive':
    if not os.path.isdir(DRIVE_INPUT_DIR):
        raise FileNotFoundError(f'Pasta de áudio não encontrada: {DRIVE_INPUT_DIR}')

    _all = sorted(
        _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.wav'),  recursive=True) +
        _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.flac'), recursive=True) +
        _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.mp3'),  recursive=True)
    )

    if FILENAME_PREFIX:
        _all = [f for f in _all if os.path.basename(f).startswith(FILENAME_PREFIX + '_')]

    if FILTER_ENABLED and _all:
        _fs = datetime.strptime(f"{FILTER_START_DATE} {FILTER_START_TIME or '00:00'}", '%Y-%m-%d %H:%M')
        _fe = datetime.strptime(f"{FILTER_END_DATE} {FILTER_END_TIME or '23:59'}",   '%Y-%m-%d %H:%M')
        _all = [
            f for f in _all
            if (lambda dt: dt is None or _fs <= dt <= _fe)(_parse_file_datetime(f, FILENAME_PREFIX))
        ]

    for f in _all:
        site = _get_site_name(f, DRIVE_INPUT_DIR)
        dt   = _parse_file_datetime(f, FILENAME_PREFIX)
        all_audio_info.append({'path': f, 'site': site, 'dt': dt})

elif AUDIO_SOURCE == 'arbimon':
    from datetime import timedelta

    _AUDIO_DIR = '/content/audio'
    os.makedirs(_AUDIO_DIR, exist_ok=True)

    try:
        _jobs = JOBS
    except NameError:
        raise RuntimeError('JOBS não definido. Execute o Parte 3 (configuração do Arbimon) primeiro.')
    if not _jobs:
        raise RuntimeError('Nenhum ponto Arbimon configurado. Preencha o Parte 3 primeiro.')

    def _parse_arbimon_dt(filename, utc_offset_hours=0, timezone_str=None):
        from datetime import timedelta
        name = os.path.basename(filename)
        utc_dt = None
        m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
        if m:
            utc_dt = datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                              int(m.group(4)), int(m.group(5)), int(m.group(6)))
        else:
            m = re.search(r'(\d{8})_(\d{6})', name)
            if m:
                d, t = m.group(1), m.group(2)
                utc_dt = datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                                  int(t[:2]), int(t[2:4]), int(t[4:6]))
        if utc_dt is None:
            return None
        if timezone_str:
            try:
                from zoneinfo import ZoneInfo
                local_dt = utc_dt.replace(tzinfo=ZoneInfo('UTC')).astimezone(ZoneInfo(timezone_str))
                return local_dt.replace(tzinfo=None)
            except Exception:
                pass
        return utc_dt + timedelta(hours=utc_offset_hours)

    _job_lookup = {}
    for _j in _jobs:
        _job_lookup[_j['stream_id']]   = _j
        _job_lookup[_j['stream_name']] = _j

    def _resolve_job(audio_path):
        job = _job_lookup.get(os.path.basename(os.path.dirname(audio_path)))
        if job:
            return job
        for _j in _jobs:
            if _j['stream_id'] and _j['stream_id'] in audio_path:
                return _j
        return _jobs[0] if len(_jobs) == 1 else {}

    for job in _jobs:
        utc_off     = job['utc_offset_hours']
        job_utc_min = job['min_date'] - timedelta(hours=utc_off)
        job_utc_max = job['max_date'] - timedelta(hours=utc_off)
        _sep = '─' * 65
        print(f"\n{_sep}")
        print(f"Ponto  : {job['stream_name']}  (id: {job['stream_id']})")
        print(f"Período: {job['min_date'].strftime('%Y-%m-%d %H:%M')} → {job['max_date'].strftime('%Y-%m-%d %H:%M')}")
        _stream_dir = os.path.join(_AUDIO_DIR, job['stream_id'])
        _existing = (
            _glob.glob(f'{_stream_dir}/**/*.wav',  recursive=True) +
            _glob.glob(f'{_stream_dir}/**/*.flac', recursive=True) +
            _glob.glob(f'{_stream_dir}/**/*.mp3',  recursive=True)
        )
        if _existing:
            print(f'  {len(_existing)} arquivo(s) já baixado(s) — pulando.')
        else:
            print('Baixando...')
            try:
                client.download_segments(
                    dest_path=_AUDIO_DIR,
                    stream=job['stream_id'],
                    min_date=job_utc_min,
                    max_date=job_utc_max,
                    parallel=True,
                )
            except Exception as e:
                print(f'  ERRO: {e}')

    _all = sorted(
        _glob.glob(f'{_AUDIO_DIR}/**/*.wav',  recursive=True) +
        _glob.glob(f'{_AUDIO_DIR}/**/*.flac', recursive=True) +
        _glob.glob(f'{_AUDIO_DIR}/**/*.mp3',  recursive=True)
    )

    for f in _all:
        _job = _resolve_job(f)
        site = _job.get('stream_name', os.path.basename(os.path.dirname(f)))
        dt   = _parse_arbimon_dt(
            os.path.basename(f),
            utc_offset_hours=_job.get('utc_offset_hours', 0),
            timezone_str=_job.get('timezone_str', None)
        )
        if dt is not None and _job.get('min_date') and _job.get('max_date'):
            if not (_job['min_date'] <= dt <= _job['max_date']):
                continue
        all_audio_info.append({'path': f, 'site': site, 'dt': dt})

# Resumo
_sites = defaultdict(list)
for info in all_audio_info:
    _sites[info['site']].append(info)

print(f'Fonte de áudio : {AUDIO_SOURCE}')
print(f'Total de arquivos  : {len(all_audio_info)}')
print(f'Pontos encontrados : {len(_sites)}')
print()
for site_name, entries in sorted(_sites.items()):
    dts = [e['dt'] for e in entries if e['dt'] is not None]
    if dts:
        date_range = f"{min(dts).strftime('%Y-%m-%d %H:%M')} → {max(dts).strftime('%Y-%m-%d %H:%M')}"
    else:
        date_range = '(sem marcação de data/hora)'
    print(f'  {site_name}')
    print(f'    Arquivos : {len(entries)}')
    print(f'    Período  : {date_range}')

if not all_audio_info:
    print('Nenhum arquivo de áudio encontrado. Verifique a configuração de fonte de áudio no Parte 2.')
else:
    print('\nVarredura concluída. Continue para o Parte 5.')

---
## Parte 5 — Carregar modelo e inicializar banco de dados (*database*)

Esta célula baixa o modelo *backbone* (se necessário) e cria o banco de dados SQLite de embeddings.

Se o banco de dados já existir, ele verifica se a configuração do modelo corresponde ao banco existente — não é possível misturar embeddings de modelos diferentes no mesmo banco.

In [ ]:
#@title 🧠 Carregar e inicializar { display-mode: "form" }

import os
import sqlite3
import numpy as np
from pathlib import Path
from datetime import datetime

# --- Resolve e carrega modelo ONNX ---
if HF_REPO:
    from huggingface_hub import hf_hub_download
    print(f'Baixando backbone do HuggingFace: {HF_REPO} / {HF_FILE} ...')
    _model_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE)
else:
    _model_path = CUSTOM_DRIVE_MODEL_PATH

if not os.path.exists(_model_path):
    raise FileNotFoundError(f'Modelo não encontrado: {_model_path}\nVerifique a configuração do modelo no Parte 2.')

import onnxruntime as ort

# Seleciona o provedor de execução com base no dispositivo escolhido no Parte 1.
_device = globals().get('COMPUTE_DEVICE', 'CPU')
_available = ort.get_available_providers()
if _device == 'GPU' and 'CUDAExecutionProvider' in _available:
    _providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
else:
    if _device == 'GPU':
        print('AVISO: GPU solicitada, mas CUDAExecutionProvider não está disponível — voltando para CPU.')
        print('       Defina o runtime do Colab como GPU (Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4)')
        print('       e execute novamente desde o Parte 1 para usar a GPU.')
    _providers = ['CPUExecutionProvider']

_ort_session = ort.InferenceSession(_model_path, providers=_providers)
_in_names    = [i.name for i in _ort_session.get_inputs()]
_out_names   = [o.name for o in _ort_session.get_outputs()]

# Valida o nome de saída; usa a primeira saída se não encontrado
if OUTPUT_NAME in _out_names:
    _fetch_output = OUTPUT_NAME
else:
    _fetch_output = _out_names[0]
    print(f"AVISO: saída '{OUTPUT_NAME}' não encontrada no modelo; usando '{_fetch_output}'.")

print('Modelo ONNX carregado.')
print(f'  Caminho    : {_model_path}')
print(f"  Entradas   : {_in_names}  →  usando '{INPUT_NAME}'")
print(f"  Saídas     : {_out_names}  →  buscando '{_fetch_output}'")
print(f'  Dispositivo: {_device}')
print(f'  Provedores : {_ort_session.get_providers()}')

# --- Inicializa banco de dados SQLite ---
_db_path = Path(DB_PATH.strip())
if DB_MODE == 'update existing':
    if not _db_path.exists():
        raise FileNotFoundError(f'Banco de dados existente não encontrado: {_db_path}')
else:
    if _db_path.exists():
        raise FileExistsError(
            f'Banco de dados já existe: {_db_path}\n'
            'Mude DB_MODE para "update existing" para adicionar gravações, '
            'ou defina um DB_PATH diferente (ou delete o arquivo) para começar do zero.'
        )
    _db_path.parent.mkdir(parents=True, exist_ok=True)

_meta = {
    'format_version':    '1',
    'model_id':          MODEL_NAME,
    'sample_rate':       str(SAMPLE_RATE),
    'window_duration_s': str(WINDOW_S),
    'embedding_size':    str(EMBEDDING_SIZE),
    'overlap':           str(SEGMENT_OVERLAP),
    'embedding_dtype':   'float32',
}
if HF_REPO:
    _meta['hf_repo'] = HF_REPO
    _meta['hf_file'] = HF_FILE

with sqlite3.connect(_db_path) as _con:
    _con.execute('CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY, value TEXT NOT NULL)')
    _con.execute('''
        CREATE TABLE IF NOT EXISTS embeddings (
            site_name    TEXT    NOT NULL DEFAULT '',
            recording_id TEXT    NOT NULL,
            chunk_index  INTEGER NOT NULL,
            datetime     TEXT,
            data         BLOB    NOT NULL,
            PRIMARY KEY (site_name, recording_id, chunk_index)
        )''')
    _con.execute('''
        CREATE INDEX IF NOT EXISTS idx_emb_recording
        ON embeddings(site_name, recording_id, chunk_index)''')
    _con.execute('''
        CREATE TABLE IF NOT EXISTS sites (
            site_name  TEXT PRIMARY KEY,
            stream_id  TEXT NOT NULL DEFAULT '',
            utc_offset REAL NOT NULL DEFAULT 0.0,
            lat        REAL,
            lon        REAL
        )''')
    _con.commit()

    _existing = dict(_con.execute('SELECT key, value FROM metadata').fetchall())
    _check_keys = ('model_id', 'sample_rate', 'window_duration_s', 'embedding_size', 'overlap')
    if _existing:
        _mismatch = [
            f"  {k}: banco='{_existing[k]}' vs atual='{_meta[k]}'"
            for k in _check_keys
            if k in _existing and _existing[k] != _meta[k]
        ]
        if _mismatch:
            print('\nERRO: o banco existente foi criado com uma configuração diferente:')
            for m in _mismatch:
                print(m)
            print('Use uma pasta de saída diferente, ou delete o arquivo .db para começar do zero.')
        else:
            _n = _con.execute('SELECT COUNT(*) FROM embeddings').fetchone()[0]
            print(f'\nBanco existente: {_n} embedding(s) já armazenado(s).')
    else:
        _con.executemany(
            'INSERT OR IGNORE INTO metadata (key, value) VALUES (?, ?)',
            list(_meta.items()) + [('created_at', datetime.now().isoformat())]
        )
        _con.commit()
        print('\nNovo banco de dados de embeddings criado.')

print(f'\nBanco de dados : {_db_path}')
for k, v in _meta.items():
    print(f'  {k}: {v}')

---
## Parte 6 — Verificar embeddings já extraídos

Esta célula verifica quais arquivos de áudio já foram processados e armazenados no banco de dados.
Esses arquivos serão **desconsiderados** na Parte 7, para que você possa executar com segurança sem extrair novamente embeddings existentes.

In [ ]:
#@title 🔎 Verificar { display-mode: "form" }

import sqlite3
import shutil
import os
from pathlib import Path

# Trabalha com uma cópia local para que a consulta seja rápida e as escritas
# subsequentes fiquem no armazenamento local (evita escritas lentas via FUSE do Drive).
_local_db = Path("/content") / _db_path.name

if _db_path.exists():
    print(f"Copiando banco do Drive para armazenamento local...")
    shutil.copy2(_db_path, _local_db)
    print(f"  {_db_path.stat().st_size / 1024:.1f} kB copiados.")
else:
    print("AVISO: banco de dados não encontrado no Drive — execute o Parte 5 primeiro.")

to_process   = []
already_done = []

with sqlite3.connect(_local_db) as _con:
    for info in all_audio_info:
        stem  = os.path.splitext(os.path.basename(info["path"]))[0]
        count = _con.execute(
            "SELECT COUNT(*) FROM embeddings WHERE site_name = ? AND recording_id = ?",
            (info['site'], stem)
        ).fetchone()[0]
        if count > 0:
            already_done.append(info)
        else:
            to_process.append(info)

print(f"Banco local         : {_local_db}")
print(f"Banco no Drive      : {_db_path}")
print(f"Total de arquivos   : {len(all_audio_info)}")
print(f"Já calculados       : {len(already_done)}")
print(f"Restam para processar: {len(to_process)}")

if already_done:
    print()
    print("Já concluídos (serão pulados):")
    for info in already_done[:10]:
        print(f"  {info['site']}/{os.path.basename(info['path'])}")
    if len(already_done) > 10:
        print(f"  ... e mais {len(already_done) - 10}")

if not to_process:
    print()
    print("Todos os arquivos já foram processados. Nada a fazer!")
else:
    print()
    print(f"Pronto para extrair embeddings de {len(to_process)} arquivo(s). Prossiga para a Parte 7.")

---
## Parte 7 (CPU) — Extrair e salvar embeddings

Esta é a etapa principal de computação (se estiver usando GPU, ignore esta parte e continue na Parte 8). Para cada arquivo de áudio:
1. A gravação é carregada e reamostrada para a taxa do modelo
2. É dividida em janelas da duração configurada (com sobreposição opcional)
3. Cada janela é processada pelo modelo backbone
4. O vetor de embedding resultante é extraído e salvo no banco de dados SQLite

> Dependendo do número de arquivos e da duração das gravações, esta etapa pode demorar.

In [ ]:
#@title 🚀 Extrair { display-mode: "form" }

import os
import shutil
import sqlite3
import time
import numpy as np
import librosa
from pathlib import Path

# Sincroniza o banco local com o Drive a cada N arquivos (0 = apenas no final).
SYNC_EVERY = 50  #@param {type:"integer"}

#@markdown **Tamanho do lote (batch)** — quantas janelas de áudio são enviadas ao modelo de uma vez.
#@markdown Lotes maiores usam a GPU de forma mais eficiente (tente 64–256 na GPU).
#@markdown Na CPU, um valor pequeno (8–16) é suficiente. Reduza se ocorrer erro de memória.
BATCH_SIZE = 8  #@param {type:"integer"}
BATCH_SIZE = max(1, int(BATCH_SIZE))

_local_db       = Path("/content") / _db_path.name
_stride_samples = max(1, int(WINDOW_SAMPLES * (1 - SEGMENT_OVERLAP)))

def _embed_batch(segs):
    """Executa a inferência em uma lista de janelas de mesmo tamanho. Retorna array (n, embedding).
    Recorre a uma janela por vez se o modelo não aceitar dimensão de lote."""
    arr = np.stack(segs).astype(np.float32)
    try:
        out = _ort_session.run([_fetch_output], {INPUT_NAME: arr})[0]
        return np.asarray(out, dtype=np.float32).reshape(len(segs), -1)
    except Exception:
        rows = [
            _ort_session.run([_fetch_output], {INPUT_NAME: s[np.newaxis, :]})[0].reshape(-1)
            for s in segs
        ]
        return np.asarray(rows, dtype=np.float32)

def _store_batch(con, segs, site_name, file_stem, rec_datetime, chunk_index):
    """Computa o embedding de um lote de janelas e os insere, retornando o próximo chunk_index."""
    embs = _embed_batch(segs)
    rows = []
    for emb in embs:
        rows.append((site_name, file_stem, chunk_index, rec_datetime, emb.tobytes()))
        chunk_index += 1
    con.executemany(
        "INSERT OR REPLACE INTO embeddings "
        "(site_name, recording_id, chunk_index, datetime, data) VALUES (?, ?, ?, ?, ?)",
        rows,
    )
    return chunk_index

def _sync_to_drive():
    shutil.copy2(_local_db, _db_path)
    size_mb = _local_db.stat().st_size / 1024 / 1024
    print(f"  [sync] {size_mb:.1f} MB → {_db_path}")

_device = globals().get('COMPUTE_DEVICE', 'CPU')

if _device == 'GPU':
    print("COMPUTE_DEVICE está definido como GPU.")
    print("Esta célula executa o caminho de CPU e permanece inalterada para runtimes de CPU.")
    print("Para a GPU, execute a próxima célula:")
    print("  'Parte 7 (GPU) — Computar embeddings (pipeline de GPU)'.")
    print("Ela sobrepõe a decodificação do áudio com a inferência e agrupa janelas entre arquivos,")
    print("o que de fato mantém a T4 ocupada.")
elif not to_process:
    print("Nenhum arquivo para processar. Todos os embeddings já foram computados.")
else:
    print(f"Computando embeddings para {len(to_process)} arquivo(s)")
    print(f"  Modelo            : {MODEL_NAME}")
    print(f"  Taxa de amostragem: {SAMPLE_RATE} Hz")
    print(f"  Janela            : {WINDOW_S} s  ({WINDOW_SAMPLES} amostras)")
    print(f"  Passo             : {_stride_samples / SAMPLE_RATE:.3f} s  ({_stride_samples} amostras)")
    print(f"  Embedding         : {EMBEDDING_SIZE}-dim")
    print(f"  Tamanho do lote   : {BATCH_SIZE}")
    print(f"  Dispositivo       : {globals().get('COMPUTE_DEVICE', 'CPU')}  ({_ort_session.get_providers()})")
    print(f"  Banco local       : {_local_db}")
    print(f"  Banco no Drive    : {_db_path}")
    print(f"  Sync Drive        : a cada {SYNC_EVERY} arquivo(s)")
    print("=" * 65)

    total_segments = 0
    total_start    = time.time()


    # Metadados por ponto (stream_id, UTC offset, coordenadas)
    _site_meta = {}
    if AUDIO_SOURCE == 'arbimon':
        _jobs_by_key = {}
        try:
            for _j in JOBS:
                _jobs_by_key[_j['stream_name']] = _j
                _jobs_by_key[_j['stream_id']]   = _j
        except NameError:
            pass
        for _info in to_process:
            _sn = _info['site']
            if _sn not in _site_meta:
                _j = _jobs_by_key.get(_sn, {})
                _site_meta[_sn] = {
                    'stream_id':  _j.get('stream_id', ''),
                    'utc_offset': float(_j.get('utc_offset_hours', 0)),
                    'lat':        _j.get('lat', '') or '',
                    'lon':        _j.get('lon', '') or '',
                }
    else:
        for _info in to_process:
            _sn = _info['site']
            if _sn not in _site_meta:
                _coords = _latlon_map.get(_sn, ('', ''))
                _site_meta[_sn] = {
                    'stream_id':  '',
                    'utc_offset': 0.0,
                    'lat':        _coords[0] if _coords[0] != '' else None,
                    'lon':        _coords[1] if _coords[1] != '' else None,
                }
    with sqlite3.connect(_local_db) as _con:
        _con.execute("PRAGMA journal_mode=MEMORY")
        _con.execute("PRAGMA synchronous=NORMAL")
        _con.execute('''
            CREATE TABLE IF NOT EXISTS sites (
                site_name  TEXT PRIMARY KEY,
                stream_id  TEXT NOT NULL DEFAULT '',
                utc_offset REAL NOT NULL DEFAULT 0.0,
                lat        REAL,
                lon        REAL
            )''')

        for _sn, _sm in _site_meta.items():
            _lat = float(_sm['lat']) if _sm['lat'] not in ('', None) else None
            _lon = float(_sm['lon']) if _sm['lon'] not in ('', None) else None
            _con.execute(
                'INSERT OR IGNORE INTO sites (site_name, stream_id, utc_offset, lat, lon) VALUES (?, ?, ?, ?, ?)',
                (_sn, _sm['stream_id'], _sm['utc_offset'], _lat, _lon)
            )

        for file_idx, info in enumerate(to_process, 1):
            audio_path = info["path"]
            site_name  = info["site"]
            filename   = os.path.basename(audio_path)
            file_stem  = os.path.splitext(filename)[0]

            print(f"[{file_idx}/{len(to_process)}] {filename}  (ponto: {site_name})")

            try:
                audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
            except Exception as e:
                print(f"  ERRO ao carregar áudio: {e} — pulando.")
                continue

            file_start   = time.time()
            n_segments   = 0
            rec_datetime = info['dt'].isoformat() if info['dt'] else None
            chunk_index  = 0

            # Acumula janelas e executa o modelo em lotes (memória limitada).
            _batch = []
            for start_samp in range(0, len(audio), _stride_samples):
                seg = audio[start_samp:start_samp + WINDOW_SAMPLES]
                if len(seg) < WINDOW_SAMPLES * 0.5:
                    continue
                if len(seg) < WINDOW_SAMPLES:
                    seg = np.pad(seg, (0, WINDOW_SAMPLES - len(seg)))
                _batch.append(seg.astype(np.float32))
                if len(_batch) >= BATCH_SIZE:
                    try:
                        chunk_index = _store_batch(_con, _batch, site_name, file_stem, rec_datetime, chunk_index)
                        n_segments += len(_batch)
                    except Exception as e:
                        print(f"  ERRO durante inferência: {e}")
                    _batch = []

            if _batch:
                try:
                    chunk_index = _store_batch(_con, _batch, site_name, file_stem, rec_datetime, chunk_index)
                    n_segments += len(_batch)
                except Exception as e:
                    print(f"  ERRO durante inferência: {e}")

            _con.commit()
            total_segments += n_segments
            elapsed     = time.time() - file_start
            audio_dur_s = len(audio) / SAMPLE_RATE
            per_seg_ms  = elapsed / n_segments * 1000 if n_segments else 0
            print(f"  → {n_segments} segmentos  |  {elapsed:.1f}s  "
                  f"({per_seg_ms:.0f} ms/seg, {audio_dur_s / elapsed:.1f}x tempo real)")

            if SYNC_EVERY > 0 and file_idx % SYNC_EVERY == 0:
                _sync_to_drive()

    # Sincronização final com o Drive
    _sync_to_drive()

    total_elapsed = time.time() - total_start
    print()
    print("=" * 65)
    print("Extração de embeddings concluída!")
    print(f"  Arquivos processados : {len(to_process)}")
    print(f"  Total de segmentos   : {total_segments}")
    print(f"  Tempo total          : {total_elapsed:.1f}s")
    print(f"  Banco de dados       : {_db_path}")
    print(f"  Tamanho do banco     : {_db_path.stat().st_size / 1024 / 1024:.2f} MB")

---
## Parte 7 (GPU) — Extrair e salvar embeddings

**Use esta célula no lugar da anterior quando você selecionou `GPU` no Parte 1.**
Em runtimes de CPU, continue usando a célula de CPU acima — ela permanece inalterada.

Em um runtime com GPU, a célula simples acima oferece pouco ganho de velocidade porque a GPU fica ociosa enquanto cada arquivo é decodificado e reamostrado na CPU, um arquivo de cada vez. Esta célula corrige isso estruturalmente:

- **A decodificação/reamostragem roda em threads de trabalho da CPU** (`NUM_LOADERS`) à frente da GPU, então a GPU nunca fica esperando pelo `librosa.load`.
- **As janelas são agrupadas entre arquivos**, então um `BATCH_SIZE` grande (128–256 em uma T4) é de fato preenchido e a GPU faz trabalho denso.
- **O modelo é testado quanto ao suporte** antecipadamente — se o backbone não tiver a opção de *batch processing*, a execução é cancelada com uma mensagem clara em vez de rodar silenciosamente uma janela por vez.

O banco de dados de resultados, os metadados e o comportamento de sincronização com o Drive são idênticos aos da Parte 7.

> **Dica:** se ainda estiver lento com `NUM_LOADERS = 4`, a decodificação do áudio é o problema, não a GPU — aumente `NUM_LOADERS` ou compare o tempo de `librosa.load` com o tempo de inferência para alguns arquivos.

In [ ]:
#@title 🚀 Extrair embeddings (pipeline de GPU) { display-mode: "form" }
#@markdown Use esta célula **no lugar da** célula de CPU acima quando você definir o dispositivo
#@markdown de computação como **GPU** no Parte 1. Ela decodifica/reamostra o áudio em threads
#@markdown de trabalho da CPU enquanto a GPU executa a inferência, e preenche lotes grandes entre
#@markdown arquivos, mantendo a T4 ocupada. A célula de CPU acima permanece inalterada — use-a em runtimes de CPU.

import os, shutil, sqlite3, time, threading, queue
import numpy as np
import librosa
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# Sincroniza o banco local com o Drive a cada N arquivos (0 = apenas no final).
SYNC_EVERY = 50   #@param {type:"integer"}

#@markdown **Tamanho do lote (batch)** — janelas enviadas à GPU de uma vez. Com o agrupamento
#@markdown entre arquivos isso agora faz diferença; tente 128–256 em uma T4. Reduza em caso de erro de memória.
BATCH_SIZE = 128  #@param {type:"integer"}

#@markdown **Threads de carregamento** — workers da CPU decodificando/reamostrando o áudio à frente da GPU.
NUM_LOADERS = 4   #@param {type:"integer"}

#@markdown **Queue depth** — quantos arquivos decodificados podem aguardar em memória pela GPU.
QUEUE_DEPTH = 8   #@param {type:"integer"}

BATCH_SIZE  = max(1, int(BATCH_SIZE))
NUM_LOADERS = max(1, int(NUM_LOADERS))

_local_db       = Path("/content") / _db_path.name
_stride_samples = max(1, int(WINDOW_SAMPLES * (1 - SEGMENT_OVERLAP)))

# --- Proteção: esta célula é apenas para o caminho de GPU ---------------------
_gpu_ready = True
if globals().get('COMPUTE_DEVICE', 'CPU') != 'GPU' or \
   'CUDAExecutionProvider' not in _ort_session.get_providers():
    print(
        "AVISO: Esta é a célula de GPU, mas os provedores ativos são "
        f"{_ort_session.get_providers()}.\n"
        "Defina COMPUTE_DEVICE='GPU' no Parte 1 (com um runtime T4) e execute novamente desde o início, "
        "ou simplesmente use a célula de CPU acima."
    )
    _gpu_ready = False

# --- Verifica se o modelo realmente aceita dimensão de lote > 1 (sem fallback silencioso) -
if _gpu_ready:
    _probe = np.zeros((2, WINDOW_SAMPLES), dtype=np.float32)
    try:
        _ort_session.run([_fetch_output], {INPUT_NAME: _probe})
    except Exception as e:
        print(
            "AVISO: Este modelo ONNX não aceita dimensão de lote > 1 "
            f"(teste falhou: {e}).\n"
            "O agrupamento — e, portanto, o ganho de velocidade da GPU — requer dimensão de lote dinâmica. "
        )
        _gpu_ready = False

if _gpu_ready:
    def _embed_batch_arr(arr):
        """Executa a inferência em um array float32 (n, WINDOW_SAMPLES) → array (n, embedding)."""
        out = _ort_session.run([_fetch_output], {INPUT_NAME: arr})[0]
        return np.asarray(out, dtype=np.float32).reshape(arr.shape[0], -1)

    def _load_file(info):
        """Worker da CPU: copia localmente, decodifica, reamostra e janela um arquivo.

        Copia para /content antes de ler, em vez de ler diretamente do
        Drive montado via FUSE: threads de carregamento lendo em paralelo
        diretamente do Drive podem quebrar o mount ("Transport endpoint is
        not connected") sob carga sustentada, especialmente em lotes grandes.
        Uma única cópia sequencial por arquivo evita sobrecarregar o Drive
        com leituras concorrentes de acesso aleatório.
        """
        os.makedirs('/content/audio_tmp', exist_ok=True)
        local_path = os.path.join('/content/audio_tmp', f"{threading.get_ident()}_{os.path.basename(info['path'])}")
        try:
            shutil.copy2(info["path"], local_path)
            audio, _ = librosa.load(local_path, sr=SAMPLE_RATE, mono=True)
        except Exception as e:
            return {"info": info, "error": str(e), "segs": [], "dur": 0.0}
        finally:
            if os.path.exists(local_path):
                os.remove(local_path)
        segs = []
        ci   = 0
        for start in range(0, len(audio), _stride_samples):
            seg = audio[start:start + WINDOW_SAMPLES]
            if len(seg) < WINDOW_SAMPLES * 0.5:
                continue
            if len(seg) < WINDOW_SAMPLES:
                seg = np.pad(seg, (0, WINDOW_SAMPLES - len(seg)))
            segs.append((ci, seg.astype(np.float32)))
            ci += 1
        return {"info": info, "error": None, "segs": segs, "dur": len(audio) / SAMPLE_RATE}

    def _sync_to_drive():
        shutil.copy2(_local_db, _db_path)
        print(f"  [sync] {_local_db.stat().st_size / 1024 / 1024:.1f} MB → {_db_path}")

    if not to_process:
        print("Nenhum arquivo para processar. Todos os embeddings já foram extraídos.")
    else:
        print(f"Extraindo embeddings para {len(to_process)} arquivo(s)  [pipeline de GPU]")
        print(f"  Modelo          : {MODEL_NAME}")
        print(f"  Janela          : {WINDOW_S} s  ({WINDOW_SAMPLES} amostras)")
        print(f"  Overlap         : {_stride_samples / SAMPLE_RATE:.3f} s")
        print(f"  Tamanho do lote : {BATCH_SIZE}   Loaders: {NUM_LOADERS}   Fila: {QUEUE_DEPTH}")
        print(f"  Providers       : {_ort_session.get_providers()}")
        print(f"  Database local  : {_local_db}")
        print(f"  Database Drive  : {_db_path}")
        print("=" * 65)

        # --- Metadados por ponto (idêntico à célula de CPU) -----------------------
        _site_meta = {}
        if AUDIO_SOURCE == 'arbimon':
            _jobs_by_key = {}
            try:
                for _j in JOBS:
                    _jobs_by_key[_j['stream_name']] = _j
                    _jobs_by_key[_j['stream_id']]   = _j
            except NameError:
                pass
            for _info in to_process:
                _sn = _info['site']
                if _sn not in _site_meta:
                    _j = _jobs_by_key.get(_sn, {})
                    _site_meta[_sn] = {
                        'stream_id':  _j.get('stream_id', ''),
                        'utc_offset': float(_j.get('utc_offset_hours', 0)),
                        'lat':        _j.get('lat', '') or '',
                        'lon':        _j.get('lon', '') or '',
                    }
        else:
            for _info in to_process:
                _sn = _info['site']
                if _sn not in _site_meta:
                    _coords = _latlon_map.get(_sn, ('', ''))
                    _site_meta[_sn] = {
                        'stream_id':  '',
                        'utc_offset': 0.0,
                        'lat':        _coords[0] if _coords[0] != '' else None,
                        'lon':        _coords[1] if _coords[1] != '' else None,
                    }

        total_segments = 0
        total_start    = time.time()
        results_q      = queue.Queue(maxsize=QUEUE_DEPTH)

        def _producer():
            # ex.map limita a concorrência a NUM_LOADERS e preserva a ordem dos arquivos.
            with ThreadPoolExecutor(max_workers=NUM_LOADERS) as ex:
                for res in ex.map(_load_file, to_process):
                    results_q.put(res)
            results_q.put(None)  # sentinela

        with sqlite3.connect(_local_db) as _con:
            _con.execute("PRAGMA journal_mode=MEMORY")
            _con.execute("PRAGMA synchronous=NORMAL")
            _con.execute('''
                CREATE TABLE IF NOT EXISTS sites (
                    site_name  TEXT PRIMARY KEY,
                    stream_id  TEXT NOT NULL DEFAULT '',
                    utc_offset REAL NOT NULL DEFAULT 0.0,
                    lat        REAL,
                    lon        REAL
                )''')
            for _sn, _sm in _site_meta.items():
                _lat = float(_sm['lat']) if _sm['lat'] not in ('', None) else None
                _lon = float(_sm['lon']) if _sm['lon'] not in ('', None) else None
                _con.execute(
                    'INSERT OR IGNORE INTO sites (site_name, stream_id, utc_offset, lat, lon) '
                    'VALUES (?, ?, ?, ?, ?)',
                    (_sn, _sm['stream_id'], _sm['utc_offset'], _lat, _lon)
                )

            # O buffer mantém (seg, (site, stem, datetime, chunk_index)) entre arquivos.
            buffer       = []
            files_loaded = 0

            def _flush(n):
                """Computa o embedding e armazena as primeiras n janelas do buffer."""
                chunk = buffer[:n]
                del buffer[:n]
                arr   = np.stack([c[0] for c in chunk]).astype(np.float32)
                embs  = _embed_batch_arr(arr)
                rows  = [
                    (key[0], key[1], key[3], key[2], embs[i].tobytes())
                    for i, (_seg, key) in enumerate(chunk)
                ]
                _con.executemany(
                    "INSERT OR REPLACE INTO embeddings "
                    "(site_name, recording_id, chunk_index, datetime, data) "
                    "VALUES (?, ?, ?, ?, ?)",
                    rows,
                )
                return len(rows)

            producer = threading.Thread(target=_producer, daemon=True)
            producer.start()

            while True:
                res = results_q.get()
                if res is None:
                    break
                info      = res["info"]
                filename  = os.path.basename(info["path"])
                file_stem = os.path.splitext(filename)[0]
                files_loaded += 1

                if res["error"]:
                    print(f"[{files_loaded}/{len(to_process)}] {filename}  "
                          f"ERRO ao carregar áudio: {res['error']} — ignorando.")
                    continue

                rec_dt = info['dt'].isoformat() if info['dt'] else None
                for ci, seg in res["segs"]:
                    buffer.append((seg, (info["site"], file_stem, rec_dt, ci)))
                    while len(buffer) >= BATCH_SIZE:
                        total_segments += _flush(BATCH_SIZE)

                print(f"[{files_loaded}/{len(to_process)}] {filename}  "
                      f"(ponto: {info['site']})  → {len(res['segs'])} embeddings a serem extraídos")

                if SYNC_EVERY > 0 and files_loaded % SYNC_EVERY == 0:
                    _con.commit()
                    _sync_to_drive()

            # Esvazia as janelas restantes (último lote parcial).
            while buffer:
                total_segments += _flush(min(BATCH_SIZE, len(buffer)))

            _con.commit()
            producer.join()

        _sync_to_drive()
        total_elapsed = time.time() - total_start
        print()
        print("=" * 65)
        print("Extração de embeddings concluída!  [pipeline de GPU]")
        print(f"  Arquivos processados : {len(to_process)}")
        print(f"  Total de segmentos   : {total_segments}")
        print(f"  Tempo total          : {total_elapsed:.1f}s  "
              f"({total_segments / total_elapsed:.0f} seg/s)")
        print(f"  Banco de dados       : {_db_path}")
        print(f"  Tamanho do banco     : {_db_path.stat().st_size / 1024 / 1024:.2f} MB")
else:
    print('Pipeline de embeddings em GPU ignorado — veja o aviso acima.')
